# TP 2 - Assistant RAG simple


Ce notebook construit un assistant RAG simple à partir de la base vectorielle V1 préparée dans `2_1_rag_db_preparation.ipynb`.

### 0.1. Objectif
- **TP 2_1** : Préparer la base vectorielle V1 (chunking par caractères, embeddings, indexation Chroma)
- **TP 2_2** : Créer un assistant RAG simple : vectoriser la question, récupérer les chunks les plus proches, générer une réponse ancrée
- **TP 2_3** : Préparer la base vectorielle V2 (chunking par en-têtes Markdown)
- **TP 2_4** : Créer un assistant RAG avec des méthodes avancées de retrieval (Multi-Query, HyDE, Reranking)

### 0.2. Documentation générale

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[ChromaDB](https://docs.trychroma.com/)

In [ ]:
from pathlib import Path
from typing import Callable

import chromadb

from shared.config import ROOT_DIR
from shared.rag_utils import RAGChunk, rag_embed_text_batch, rag_embed_text_batch_local

# INFO : Choix LLM entre local ou cloud (fonction fournies dans le dossier shared/llm_utils.py)
from shared.llm_utils import (
    LLMRequest,
    run_llm, # Cloud
    #run_llm_local as run_llm, # Local
)

# INFO : Choix embeddings entre local ou cloud (fonction fournies dans le dossier shared/rag_utils.py)
rag_embed_fn = rag_embed_text_batch # Cloud
#rag_embed_fn = rag_embed_text_batch_local # Local

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v1"  # choisir entre chroma_db_rag_v1 et chroma_db_rag_v2

### 0.3. Use case principal
Votre objectif est de répondre à une question

In [ ]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

### 0.4. Récapitulatif des fonctions utilisées dans ce notebook

**Fonctions et classes à utiliser**

- `run_llm` : fonction qui lit la configuration, envoie la requête au modèle et retourne un `LLMResponse`
- `LLMRequest` : classe qui représente les données d'entrée d'un appel LLM
- `LLMResponse` : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)

--> Disponibles dans `shared/llm_utils.py`

- `rag_embed_text_batch` : fonction qui calcule les embeddings d'une liste de textes via l'API Google GenAI
- `rag_embed_text_batch_local` : version locale de `rag_embed_text_batch` (LMStudio ou Ollama)

--> Disponibles dans `shared/rag_utils.py`

**Fonctions corrigées dans ce notebook**

- `RAGAssistant` : classe qui implémente la recherche vectorielle sur base Chroma persistée

--> À implémenter ici, puis à copier dans `shared/rag_utils.py`

---
## 1. Créer l'assistant RAG

Chaque chunk indexé dans Chroma est représenté par un **vecteur d'embedding**.

Lors d'une recherche, **la requête est elle aussi vectorisée**, puis on calcule la similarité cosinus entre ce vecteur et tous les vecteurs de la base.

La **similarité cosinus** mesure l'angle entre deux vecteurs : une valeur proche de 1 indique que les deux textes sont alignés donc sémantiquement proches, même s'ils n'ont pas de mots en commun.

### 1.1. Recherche vectorielle corrigée

In [ ]:
class RAGAssistant:
    """Assistant de recherche vectorielle sur base Chroma persistée

    Champs
    - persist_dir : dossier de base vectorielle
    - top_k : nombre de résultats par défaut
    - embed_fn : fonction d'embedding (cloud ou local)
    - collection : collection Chroma `chunks`
    """

    def __init__(
        self,
        persist_dir: Path,
        top_k: int = 10,
        embed_fn: Callable[[list[str]], list[list[float]]] = rag_embed_text_batch,
    ) -> None:
        self.persist_dir: Path = persist_dir
        self.top_k: int = top_k
        self.embed_fn: Callable[[list[str]], list[list[float]]] = embed_fn
        if not self.persist_dir.exists():
            raise ValueError(f"persist_dir does not exist: {self.persist_dir}")

        client = chromadb.PersistentClient(path=str(self.persist_dir))
        self.collection: chromadb.Collection = client.get_collection(name="chunks")

    def search(self, query: str, top_k: int | None = None) -> list[tuple[RAGChunk, float]]:
        """Rechercher les chunks les plus proches pour une requête

        Entrées
        - query : texte utilisateur
        - top_k : surcharge optionnelle du `top_k` par défaut

        Sortie
        - liste de paires `(RAGChunk, score)`
          - `RAGChunk` : chunk retrouvé (source, chunk_id, text)
          - `score` : score de similarité dérivé de la distance vectorielle
        """
        k = top_k if top_k is not None else self.top_k
        query_embedding = self.embed_fn([f"query: {query}"])[0]
        results = self.collection.query(query_embeddings=[query_embedding], n_results=k)

        chunks_with_scores: list[tuple[RAGChunk, float]] = []
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0] if results["distances"] else [0.0] * len(documents)

        for index, document_text in enumerate(documents):
            metadata = metadatas[index]
            distance = float(distances[index])
            score = 1 / (1 + distance)

            chunk_id_text = str(metadata["chunk_id"])
            chunk_id_value: int | str = int(chunk_id_text) if chunk_id_text.isdigit() else chunk_id_text
            chunk = RAGChunk(
                source=str(metadata["source"]),
                chunk_id=chunk_id_value,
                text=str(document_text),
            )
            chunks_with_scores.append((chunk, score))

        return chunks_with_scores

### 1.2. Initialiser l'assistant

In [ ]:
rag_assistant = RAGAssistant(persist_dir=VECTOR_DB_DIR, top_k=10, embed_fn=rag_embed_fn)

---
## 2. Prompt Engineering

### 2.1. Rédiger le system prompt

Écrivez un **prompt système** similaire à celui du TP1, mais adapté à la méthode RAG.

Concrètement, il faut ajouter des instructions spécifiques à la gestion du contexte injecté, entre autres :
- **Interdire toute invention**, utiliser uniquement les informations fournies dans le contexte
- **Mentionner** clairement les **informations manquantes** pour répondre à la requête
- (optionnel) **Sourcer** chaque information

In [ ]:
system_prompt = """
Tu es un assistant de planification de voyage basé sur la méthode RAG.

### Règles :
- Utiliser uniquement les faits présents dans CONTEXTE.
- Ne jamais inventer prix, dates, horaires, adresses ou transports.
- Si une information manque, écrire: "Je ne sais pas à partir du contexte fourni."
- Citer chaque affirmation factuelle au format [source - chunk id].

### Contraintes :
- Être concis mais précis.
- Pas de mise en forme Markdown décorative.
- Pour chaque recommandation: 1 raison courte + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des suggestions concrètes aux phrases vagues.

### Format de réponse :
1) Résumé: réponse courte (2 à 4 phrases)
2) Plan: séquence pratique adaptée à la demande
3) Informations manquantes: liste concise des faits absents
"""

---
## 3. Récupérer le contexte (Retrieval)

Inspecter les chunks récupérés avant génération.

À vérifier :
- **Pertinence** : les chunks répondent-ils réellement à la requête ?
- **Couverture** : proviennent-ils de plusieurs sources utiles ?

### 3.1. Lancer la recherche de chunks

In [ ]:
top_chunks = rag_assistant.search(query=user_query, top_k=10)

### 3.2. Afficher la répartition par source

In [ ]:
print(f"Chunks récupérés : {len(top_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in top_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(top_chunks, start=1):
    preview = chunk.text.replace("\n", " ")
    print(f"Chunk #{rank} | score_distance_vecteur={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(preview)
    print("\n\n")

---
## 4. Générer la réponse

Les chunks sont concaténés dans un bloc CONTEXTE, injectés dans le prompt système, puis envoyés au modèle.

Validation concrète de la sortie :
- chaque fait important doit être sourcé,
- aucune invention,
- les zones d'incertitude doivent être annoncées explicitement.

### 4.1. Assembler le contexte et générer la réponse

In [ ]:
top_chunks = rag_assistant.search(user_query)
context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in top_chunks]
)
grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{context}"
run_result = await run_llm(LLMRequest(system_prompt=grounded_system_prompt, user_prompt=user_query))

token_usage = {
    "input_tokens": run_result.input_tokens,
    "output_tokens": run_result.output_tokens,
    "total_tokens": run_result.total_tokens,
}

print(run_result.output)
print()
print(
    f"tokens : entrée={token_usage['input_tokens']} | "
    f"sortie={token_usage['output_tokens']} | "
    f"total={token_usage['total_tokens']}"
)
